# Daily Challenge: MCP + Airbnb mini-agent

A tiny agent that combines **two MCP servers** behind one planner:

```
                      ┌──────────── notes__add_note / notes__list_notes ──► Notes MCP server (local FastMCP, notes.json)
prompt ─► planner ────┤
 (stub or real LLM)   └──────────── airbnb__airbnb_search ───────────────► Airbnb MCP server (stub, or real @openbnb/mcp-server-airbnb)
     ▲                                     │
     └──────────── tool results ◄──────────┘   (loop until the planner stops calling tools)
```

**Flow:** connect both servers over STDIO → get tool schemas → convert to LLM function specs (namespaced `server__tool`) → plan `tool_calls` → route each call to its server by prefix → execute → feed results back → print listings + saved note.

## Install

In [1]:
%pip install -q "mcp[cli]<2" nest_asyncio openai
# Optional, real Airbnb MCP server (needs Node.js / npm):  npx -y @openbnb/mcp-server-airbnb

Note: you may need to restart the kernel to use updated packages.


## Config
- **Config task (done):** `MCP_HTTP_TOKEN` is only needed when an MCP server is exposed over **HTTP** with bearer auth. Both servers here use **STDIO** (child processes, no network), so we leave it empty.
- `USE_REAL_AIRBNB`: `False` uses the stub server (fixed listings). `True` spawns the real `@openbnb/mcp-server-airbnb` with `npx`.
- `USE_REAL_LLM`: `False` uses the deterministic stub planner. `True` calls `gpt-4o-mini` on GitHub Models (needs `GITHUB_TOKEN`).

In [2]:
import os, json, re, shutil, asyncio
import nest_asyncio
nest_asyncio.apply()                                  # lets asyncio.run() work inside notebooks too

MCP_HTTP_TOKEN = os.environ.get("MCP_HTTP_TOKEN", "")   # only for HTTP transports with auth; unused with STDIO
GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN", "")
if not GITHUB_TOKEN:
    try:
        from google.colab import userdata
        GITHUB_TOKEN = userdata.get("GITHUB_TOKEN") or ""
    except Exception:
        pass

USE_REAL_AIRBNB = False          # set True to use the real Airbnb MCP server via npx
USE_REAL_LLM = bool(GITHUB_TOKEN)
LLM_MODEL, LLM_ENDPOINT = "openai/gpt-4o-mini", "https://models.github.ai/inference"
NPX_AVAILABLE = shutil.which("npx") is not None

print(f"Airbnb: {'REAL (npx)' if USE_REAL_AIRBNB else 'STUB'} | LLM: {'REAL ' + LLM_MODEL if USE_REAL_LLM else 'STUB planner'} "
      f"| npx available: {NPX_AVAILABLE} | MCP_HTTP_TOKEN set: {bool(MCP_HTTP_TOKEN)}")

Airbnb: STUB | LLM: STUB planner | npx available: True | MCP_HTTP_TOKEN set: False


## Local Notes MCP server (`notes_server.py`)

In [3]:
%%writefile notes_server.py
import json, logging, sys
from datetime import datetime
from pathlib import Path
from mcp.server.fastmcp import FastMCP

logging.basicConfig(level=logging.INFO, stream=sys.stderr, format="[notes] %(message)s")
NOTES_FILE = Path("notes.json")          # persisted: each STDIO session starts a fresh server process
mcp = FastMCP("Notes")


def _load():
    return json.loads(NOTES_FILE.read_text()) if NOTES_FILE.exists() else []


@mcp.tool()
def add_note(text: str, tag: str = "general") -> dict:
    """Save a short note (e.g. a shortlist of places to stay) with an optional tag."""
    notes = _load()
    note = {"id": len(notes) + 1, "tag": tag, "text": text, "created": datetime.now().isoformat(timespec="seconds")}
    notes.append(note)
    NOTES_FILE.write_text(json.dumps(notes, indent=2))
    logging.info("note %s saved (%s)", note["id"], tag)
    return note


@mcp.tool()
def list_notes(tag: str = "") -> list[dict]:
    """List saved notes, optionally filtered by tag."""
    return [n for n in _load() if not tag or n["tag"] == tag]


if __name__ == "__main__":
    mcp.run()

Overwriting notes_server.py


## Stub Airbnb MCP server (`airbnb_stub_server.py`), same tool names and arguments as the real one

In [4]:
%%writefile airbnb_stub_server.py
import json, logging, sys
from mcp.server.fastmcp import FastMCP

logging.basicConfig(level=logging.INFO, stream=sys.stderr, format="[airbnb-stub] %(message)s")
mcp = FastMCP("airbnb-stub")

LISTINGS = {
    "lisbon": [("Sunny loft in Alfama", 96, 4.91), ("Bairro Alto studio with balcony", 82, 4.78), ("Belém riverside flat", 120, 4.88)],
    "paris": [("Montmartre artist studio", 135, 4.86), ("Marais 1-bedroom", 168, 4.93), ("Canal Saint-Martin flat", 118, 4.71)],
    "barcelona": [("Gràcia terrace apartment", 110, 4.84), ("Eixample design flat", 142, 4.9), ("Barceloneta beach studio", 99, 4.65)],
    "tel aviv": [("Neve Tzedek boutique loft", 155, 4.89), ("Rothschild Blvd apartment", 140, 4.8), ("Jaffa sea-view studio", 112, 4.75)],
}


@mcp.tool()
def airbnb_search(location: str, adults: int = 1, checkin: str = "", checkout: str = "") -> str:
    """Search Airbnb listings for a location (stub: fixed sample data). Returns JSON text like the real server."""
    city = next((c for c in LISTINGS if c in location.lower()), None)
    logging.info("search %r -> %s", location, city)
    if city is None:
        return json.dumps({"searchResults": [], "message": f"No stub data for {location!r}; try {list(LISTINGS)}"})
    results = [{"id": f"stub-{city[:3]}-{i}", "url": f"https://www.airbnb.com/rooms/stub-{city[:3]}-{i}",
                "name": name, "price_per_night_usd": price, "rating": rating, "guests": adults}
               for i, (name, price, rating) in enumerate(LISTINGS[city], 1)]
    return json.dumps({"searchResults": results, "checkin": checkin, "checkout": checkout})


@mcp.tool()
def airbnb_listing_details(id: str) -> str:
    """Details for one listing id (stub)."""
    return json.dumps({"id": id, "details": "Stub listing: wifi, kitchen, self check-in."})


if __name__ == "__main__":
    mcp.run()

Overwriting airbnb_stub_server.py


## Client helpers: connect both servers, collect namespaced tool specs, route by prefix

In [5]:
from contextlib import AsyncExitStack
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

def server_configs():
    airbnb = (StdioServerParameters(command="npx", args=["-y", "@openbnb/mcp-server-airbnb"], env=dict(os.environ))
              if USE_REAL_AIRBNB else StdioServerParameters(command="mcp", args=["run", "airbnb_stub_server.py"]))
    return {"notes": StdioServerParameters(command="mcp", args=["run", "notes_server.py"]), "airbnb": airbnb}

def convert_to_llm_tool(prefix, tool):
    schema = tool.inputSchema or {"type": "object", "properties": {}}
    return {"type": "function", "function": {
        "name": f"{prefix}__{tool.name}",                                  # namespaced: server__tool
        "description": f"[{prefix}] {tool.description or tool.name}",
        "parameters": {"type": "object", "properties": schema.get("properties", {}), "required": schema.get("required", [])}}}

class MultiServerClient:
    def __init__(self):
        self.stack, self.sessions, self.functions = AsyncExitStack(), {}, []

    async def __aenter__(self):
        errlog = self.stack.enter_context(open("servers.log", "a"))      # server logs (stderr) -> file
        for prefix, params in server_configs().items():
            read, write = await self.stack.enter_async_context(stdio_client(params, errlog=errlog))
            session = await self.stack.enter_async_context(ClientSession(read, write))
            info = await session.initialize()
            tools = (await session.list_tools()).tools
            self.sessions[prefix] = session
            self.functions += [convert_to_llm_tool(prefix, t) for t in tools]
            print(f"🔌 connected '{prefix}' -> {info.serverInfo.name}: tools {[t.name for t in tools]}")
        return self

    async def __aexit__(self, *exc):
        await self.stack.aclose()

    async def call(self, namespaced_name, args):
        prefix, tool = namespaced_name.split("__", 1)                    # route by prefix
        result = await self.sessions[prefix].call_tool(tool, arguments=args)
        # FastMCP puts the typed return value in structuredContent["result"] (lists are otherwise split into
        # one content item per element); servers without it (e.g. the Node Airbnb server) return text content.
        if result.structuredContent and "result" in result.structuredContent:
            value = result.structuredContent["result"]
            text = value if isinstance(value, str) else json.dumps(value)
        else:
            text = "\n".join(c.text for c in result.content if hasattr(c, "text"))
        return text, result.isError

## Planners: stub (deterministic) and real LLM (GitHub Models)
The stub planner behaves like a tool-using LLM. It reads the prompt and the conversation so far, then returns either `tool_calls` or a final answer:
1. no search done yet → `airbnb__airbnb_search(location, adults)`;
2. search results present and the user asked to save/note → `notes__add_note(text=<top-2 shortlist>, tag="trip")`;
3. otherwise → a final answer summarising the results.

In [6]:
def _parse_trip(prompt):
    loc = re.search(r"\bin ([A-Z][\w\s'-]+?)(?:\s+for|\s+and|[,.?!]|$)", prompt)
    adults = re.search(r"(\d+)\s*(?:adults?|people|guests|persons)", prompt, re.I)
    return (loc.group(1).strip() if loc else "Lisbon"), int(adults.group(1)) if adults else 2

def _listings_from(messages):
    for m in reversed(messages):
        if m["role"] == "tool" and m["name"].startswith("airbnb__airbnb_search"):
            return json.loads(m["content"]).get("searchResults", [])
    return None

def stub_planner(messages, functions):
    prompt = next(m["content"] for m in messages if m["role"] == "user")
    names = {f["function"]["name"] for f in functions}
    listings = _listings_from(messages)
    note_done = any(m["role"] == "tool" and m["name"] == "notes__add_note" for m in messages)
    location, adults = _parse_trip(prompt)
    if listings is None and "airbnb__airbnb_search" in names:
        return {"tool_calls": [("airbnb__airbnb_search", {"location": location, "adults": adults})]}
    if listings and not note_done and re.search(r"\b(note|save|remember)\b", prompt, re.I):
        top = sorted(listings, key=lambda l: (-l.get("rating", 0), l.get("price_per_night_usd", 1e9)))[:2]
        text = f"{location} shortlist for {adults}: " + "; ".join(
            f"{l['name']} (${l.get('price_per_night_usd', '?')}/night, ★{l.get('rating', '?')}) {l['url']}" for l in top)
        return {"tool_calls": [("notes__add_note", {"text": text, "tag": "trip"})]}
    return {"content": f"Found {len(listings or [])} listings in {location}" + (" and saved a shortlist note." if note_done else ".")}

def real_planner(messages, functions):
    from openai import OpenAI
    client = OpenAI(base_url=LLM_ENDPOINT, api_key=GITHUB_TOKEN)
    resp = client.chat.completions.create(model=LLM_MODEL, messages=messages, tools=functions, tool_choice="auto")
    msg = resp.choices[0].message
    if msg.tool_calls:
        return {"tool_calls": [(tc.function.name, json.loads(tc.function.arguments or "{}")) for tc in msg.tool_calls],
                "raw": msg}
    return {"content": msg.content}

## Orchestrate: plan → execute → feed back, until the planner gives a final answer

In [7]:
SYSTEM = ("You are a travel mini-agent. Use airbnb__airbnb_search to find stays, then save a concise shortlist "
          "(name, price, rating, url of the best 2) with notes__add_note when the user asks to save/note. Then answer briefly.")

async def orchestrate(prompt, max_steps=5):
    async with MultiServerClient() as client:
        print("🧰 LLM function specs:", [f["function"]["name"] for f in client.functions])
        messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": prompt}]
        planner = real_planner if USE_REAL_LLM else stub_planner
        for step in range(1, max_steps + 1):
            decision = planner(messages, client.functions)
            if "tool_calls" not in decision:
                print(f"\n💬 FINAL ANSWER: {decision['content']}")
                break
            calls = decision["tool_calls"]
            messages.append({"role": "assistant", "content": None, "tool_calls": [
                {"id": f"call_{step}_{i}", "type": "function", "function": {"name": n, "arguments": json.dumps(a)}}
                for i, (n, a) in enumerate(calls)]})
            for i, (name, args) in enumerate(calls):
                print(f"\n🛠️  step {step}: tool_call {name}({args})")
                text, is_error = await client.call(name, args)
                print(("   ❌ " if is_error else "   ✅ ") + text[:300].replace("\n", " ") + ("…" if len(text) > 300 else ""))
                messages.append({"role": "tool", "tool_call_id": f"call_{step}_{i}", "name": name, "content": text})
        notes, _ = await client.call("notes__list_notes", {"tag": "trip"})
        return messages, json.loads(notes) if notes.strip().startswith("[") else notes

## Demo
**Prompt task (done):** change the city, the number of guests or the note request to test other flows. The stub has Lisbon, Paris, Barcelona and Tel Aviv.

In [8]:
import pandas as pd
if os.path.exists("notes.json"):
    os.remove("notes.json")                      # fresh demo

prompt = "Find an Airbnb in Lisbon for 2 adults and save a note with the best options."
messages, notes = asyncio.run(orchestrate(prompt))

listings = _listings_from(messages) or []
print("\n🏠 LISTINGS"); display(pd.DataFrame(listings)[["name", "price_per_night_usd", "rating", "url"]])
print("📝 SAVED NOTES (from the Notes MCP server)"); display(pd.DataFrame(notes))

🔌 connected 'notes' -> Notes: tools ['add_note', 'list_notes']


🔌 connected 'airbnb' -> airbnb-stub: tools ['airbnb_search', 'airbnb_listing_details']
🧰 LLM function specs: ['notes__add_note', 'notes__list_notes', 'airbnb__airbnb_search', 'airbnb__airbnb_listing_details']

🛠️  step 1: tool_call airbnb__airbnb_search({'location': 'Lisbon', 'adults': 2})
   ✅ {"searchResults": [{"id": "stub-lis-1", "url": "https://www.airbnb.com/rooms/stub-lis-1", "name": "Sunny loft in Alfama", "price_per_night_usd": 96, "rating": 4.91, "guests": 2}, {"id": "stub-lis-2", "url": "https://www.airbnb.com/rooms/stub-lis-2", "name": "Bairro Alto studio with balcony", "price_…

🛠️  step 2: tool_call notes__add_note({'text': 'Lisbon shortlist for 2: Sunny loft in Alfama ($96/night, ★4.91) https://www.airbnb.com/rooms/stub-lis-1; Belém riverside flat ($120/night, ★4.88) https://www.airbnb.com/rooms/stub-lis-3', 'tag': 'trip'})
   ✅ {   "id": 1,   "tag": "trip",   "text": "Lisbon shortlist for 2: Sunny loft in Alfama ($96/night, ★4.91) https://www.airbnb.com/rooms/stub-lis-1


🏠 LISTINGS


,name,price_per_night_usd,rating,url
0,Sunny loft in Alfama,96,4.91,https://www.airbnb.com/rooms/stub-lis-1
1,Bairro Alto studio with balcony,82,4.78,https://www.airbnb.com/rooms/stub-lis-2
2,Belém riverside flat,120,4.88,https://www.airbnb.com/rooms/stub-lis-3


📝 SAVED NOTES (from the Notes MCP server)


,id,tag,text,created
0,1,trip,Lisbon shortlist for 2: Sunny loft in Alfama (...,2026-10-12T13:13:12


In [9]:
# Another prompt: different city, no note requested -> the planner stops after the search
messages2, _ = asyncio.run(orchestrate("Show me places in Paris for 3 guests."))
display(pd.DataFrame(_listings_from(messages2))[["name", "price_per_night_usd", "rating"]])

🔌 connected 'notes' -> Notes: tools ['add_note', 'list_notes']


🔌 connected 'airbnb' -> airbnb-stub: tools ['airbnb_search', 'airbnb_listing_details']
🧰 LLM function specs: ['notes__add_note', 'notes__list_notes', 'airbnb__airbnb_search', 'airbnb__airbnb_listing_details']

🛠️  step 1: tool_call airbnb__airbnb_search({'location': 'Paris', 'adults': 3})
   ✅ {"searchResults": [{"id": "stub-par-1", "url": "https://www.airbnb.com/rooms/stub-par-1", "name": "Montmartre artist studio", "price_per_night_usd": 135, "rating": 4.86, "guests": 3}, {"id": "stub-par-2", "url": "https://www.airbnb.com/rooms/stub-par-2", "name": "Marais 1-bedroom", "price_per_night_…

💬 FINAL ANSWER: Found 3 listings in Paris.


,name,price_per_night_usd,rating
0,Montmartre artist studio,135,4.86
1,Marais 1-bedroom,168,4.93
2,Canal Saint-Martin flat,118,4.71


## Real Airbnb MCP server check (`@openbnb/mcp-server-airbnb` via `npx`)
We spawn the **real** server over STDIO and list its tool schemas. These are the same names the stub mimics: `airbnb_search`, `airbnb_listing_details`.

The real server **respects Airbnb's `robots.txt` by default**, and Airbnb's robots.txt disallows the search path, so a default search is refused. That refusal is shown below. The package offers an `--ignore-robots-txt` flag; we do **not** use it here because it would scrape Airbnb against its published crawling policy. To use real data, rely on an official API or partner access, or decide for yourself, for personal and occasional use, whether that flag is appropriate.

In [10]:
async def real_airbnb_check():
    if not NPX_AVAILABLE:
        print("npx not available in this runtime: skipping the real Airbnb server check.")
        return
    params = StdioServerParameters(command="npx", args=["-y", "@openbnb/mcp-server-airbnb"], env=dict(os.environ))
    with open("servers.log", "a") as errlog:
        async with stdio_client(params, errlog=errlog) as (read, write):
            async with ClientSession(read, write) as session:
                info = await session.initialize()
                tools = (await session.list_tools()).tools
                print(f"Real server: {info.serverInfo.name} {info.serverInfo.version}")
                for t in tools:
                    print(f"  - {t.name}: args {list(t.inputSchema.get('properties', {}))}")
                res = await session.call_tool("airbnb_search", {"location": "Lisbon, Portugal", "adults": 2})
                print("\nairbnb_search (robots.txt respected) -> isError:", res.isError)
                print(res.content[0].text[:500])

asyncio.run(real_airbnb_check())

Real server: airbnb 0.3.0
  - airbnb_search: args ['location', 'placeId', 'checkin', 'checkout', 'adults', 'children', 'infants', 'pets', 'minPrice', 'maxPrice', 'cursor', 'propertyType', 'ignoreRobotsText']
  - airbnb_listing_details: args ['id', 'checkin', 'checkout', 'adults', 'children', 'infants', 'pets', 'ignoreRobotsText']



airbnb_search (robots.txt respected) -> isError: True
{
  "error": "This path is disallowed by Airbnb's robots.txt to this User-agent. You may or may not want to run the server with '--ignore-robots-txt' args",
  "url": "https://www.airbnb.com/s/Lisbon--Portugal/homes?ne_lat=39.4788742&ne_lng=-8.6021946&sw_lat=38.5120014&sw_lng=-9.6801930&adults=2&children=0&infants=0&pets=0",
  "suggestion": "Consider enabling 'ignore_robots_txt' in extension settings if needed for testing"
}


## What this run shows
- **Connect:** both servers are spawned over STDIO (`notes` → `Notes`, `airbnb` → `airbnb-stub`), each with its own `ClientSession`.
- **Schemas:** the tools are converted into namespaced function specs `notes__add_note`, `notes__list_notes`, `airbnb__airbnb_search`, `airbnb__airbnb_listing_details`.
- **Plan → execute:**
  - step 1: `airbnb__airbnb_search({'location': 'Lisbon', 'adults': 2})` returns 3 listings;
  - step 2: `notes__add_note(...)` saves a shortlist of the two best-rated listings;
  - the planner then returns a final answer.

  Each call is **routed by its prefix** to the right server.
- **Results:** the listings table and the note persisted by the Notes server (read back with `list_notes`). A second prompt (Paris, 3 guests, no note) stops after the search.
- **Real server:** the real Airbnb MCP server connects and exposes the same tools. With `robots.txt` respected, its search is refused, so the stub is used for listings. With `GITHUB_TOKEN`, the same loop is driven by `gpt-4o-mini` instead of the stub planner.